# RecordDiff - Mask-Channel Audit on MIMIC-IV v3.1

## 1 · Setup

In [ ]:
!pip -q install -U duckdb
import os, time, zipfile, json, warnings, numpy as np, pandas as pd, duckdb
warnings.filterwarnings("ignore")
print("duckdb", duckdb.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Configuration

In [ ]:
# paths
ZIP_PATH   = "..."
RAW_DIR    = "..."
CACHE_NPZ  = "..."
USE_CACHE  = False

# cohort
T             = 48                         # hourly bins
WINDOW_HOURS  = 48                         # first-48h observation window
MIN_LOS_DAYS  = 2.0                        # >= 48h ICU stay
MIN_AGE       = 18
LONG_STAY_DAYS = 7.0                       # 'long_stay' label threshold

# audit
MARGIN    = 0.55
N_SPLITS  = 15
TEST_FRAC = 0.30
C_REG     = 0.5

# DuckDB resource limits
DUCK_THREADS = 2
DUCK_MEMORY  = "8GB"

# variable panel: (name, class, source, [itemids]).  class in {protocol, acuity, triggered}
PANEL = [
    ("HeartRate",       "protocol",  "chart", [220045]),
    ("RespRate",        "protocol",  "chart", [220210]),
    ("SpO2",            "protocol",  "chart", [220277]),
    ("NBP_mean",        "protocol",  "chart", [220181]),
    ("NBP_sys",         "protocol",  "chart", [220179]),
    ("NBP_dia",         "protocol",  "chart", [220180]),
    ("Temperature_F",   "protocol",  "chart", [223761]),
    ("GCS_eye",         "acuity",    "chart", [220739]),
    ("GCS_verbal",      "acuity",    "chart", [223900]),
    ("GCS_motor",       "acuity",    "chart", [223901]),
    ("Creatinine",      "acuity",    "lab",   [50912]),
    ("BUN",             "acuity",    "lab",   [51006]),
    ("Sodium",          "acuity",    "lab",   [50983]),
    ("Potassium",       "acuity",    "lab",   [50971]),
    ("Chloride",        "acuity",    "lab",   [50902]),
    ("Bicarbonate",     "acuity",    "lab",   [50882]),
    ("Glucose_lab",     "acuity",    "lab",   [50931]),
    ("Hematocrit",      "acuity",    "lab",   [51221]),
    ("Hemoglobin",      "acuity",    "lab",   [51222]),
    ("WBC",             "acuity",    "lab",   [51301]),
    ("Platelets",       "acuity",    "lab",   [51265]),
    ("Magnesium",       "acuity",    "lab",   [50960]),
    ("Calcium",         "acuity",    "lab",   [50893]),
    ("Phosphate",       "acuity",    "lab",   [50970]),
    ("Lactate",         "triggered", "lab",   [50813]),
    ("Troponin_T",      "triggered", "lab",   [51003]),
    ("pH_bg",           "triggered", "lab",   [50820]),
    ("pO2_bg",          "triggered", "lab",   [50821]),
    ("pCO2_bg",         "triggered", "lab",   [50818]),
    ("BaseExcess",      "triggered", "lab",   [50802]),
    ("Bilirubin_total", "triggered", "lab",   [50885]),
    ("INR",             "triggered", "lab",   [51237]),
    ("PT",              "triggered", "lab",   [51274]),
    ("PTT",             "triggered", "lab",   [51275]),
    ("Albumin",         "triggered", "lab",   [50862]),
    ("CRP",             "triggered", "lab",   [50889]),
]

VAR_NAMES  = [p[0] for p in PANEL]
VAR_CLASS  = np.array([p[1] for p in PANEL])
V          = len(PANEL)
item2var   = {iid: i for i, p in enumerate(PANEL) for iid in p[3]}
CHART_IDS  = sorted({iid for p in PANEL if p[2] == "chart" for iid in p[3]})
LAB_IDS    = sorted({iid for p in PANEL if p[2] == "lab"   for iid in p[3]})
print(f"panel: V={V} variables | {len(CHART_IDS)} chart itemids | {len(LAB_IDS)} lab itemids")
print("classes:", {c: int((VAR_CLASS == c).sum()) for c in ['protocol', 'acuity', 'triggered']})

## 3 · Load cached cohort if it exists

In [ ]:
COHORT_READY = False
if USE_CACHE and os.path.exists(CACHE_NPZ):
    d = np.load(CACHE_NPZ, allow_pickle=True)
    m = d["m"]; y = d["y"]
    labels_tbl = pd.DataFrame({k[4:]: d[k] for k in d.files if k.startswith("lab_")})
    VAR_NAMES = list(d["var_names"]); VAR_CLASS = d["var_class"]; V = m.shape[2]
    C_COV = d["c"] if "c" in d.files else None
    COV_NAMES = list(d["cov_names"]) if "cov_names" in d.files else None
    COHORT_READY = True
    print(f"Loaded cache {CACHE_NPZ}")
    print(f"  m,y shape = {m.shape} | tasks = {list(labels_tbl.columns)}")
    print("  prevalences:", {c: round(labels_tbl[c].mean(), 4) for c in labels_tbl.columns})
else:
    print("No cache found (or USE_CACHE=False) -> run extraction cells 4–7.")

## 4 · Extract the files we need from the archive

In [ ]:
if not COHORT_READY:
    os.makedirs(RAW_DIR, exist_ok=True)
    NEEDED = {
        "patients":   "hosp/patients.csv.gz",
        "admissions": "hosp/admissions.csv.gz",
        "labevents":  "hosp/labevents.csv.gz",
        "d_labitems": "hosp/d_labitems.csv.gz",
        "icustays":   "icu/icustays.csv.gz",
        "chartevents":"icu/chartevents.csv.gz",
        "d_items":    "icu/d_items.csv.gz",
    }
    assert os.path.exists(ZIP_PATH), f"Archive not found at {ZIP_PATH}"
    zf = zipfile.ZipFile(ZIP_PATH)
    names = zf.namelist()
    FP = {}
    for key, suffix in NEEDED.items():
        hits = [n for n in names if n.endswith(suffix)]
        if not hits:
            raise FileNotFoundError(
                f"Could not find '{suffix}' in archive.\nArchive members (first 40):\n"
                + "\n".join(names[:40]))
        member = hits[0]
        dest = os.path.join(RAW_DIR, os.path.basename(member))
        if os.path.exists(dest) and os.path.getsize(dest) > 0:
            print(f"  [skip] {os.path.basename(member)} already extracted")
        else:
            t0 = time.time()
            with zf.open(member) as src, open(dest, "wb") as out:
                while True:
                    chunk = src.read(1 << 22)
                    if not chunk:
                        break
                    out.write(chunk)
            print(f"  [ok]   {os.path.basename(member)}  "
                  f"({os.path.getsize(dest)/1e6:.0f} MB, {time.time()-t0:.0f}s)")
        FP[key] = dest
    zf.close()
    print("extraction complete ->", RAW_DIR)

## 5 · Build the cohort and stay-level labels

In [ ]:
if not COHORT_READY:
    con = duckdb.connect()
    con.execute(f"PRAGMA threads={DUCK_THREADS}; PRAGMA memory_limit='{DUCK_MEMORY}';")
    con.execute(f"PRAGMA temp_directory='{RAW_DIR}/duck_tmp';")

    def rc(path):
        return (f"read_csv('{path}', header=true, sep=',', all_varchar=true, "
                f"compression='gzip', ignore_errors=true)")

    cohort = con.execute(f"""
        WITH ic AS (
            SELECT subject_id, hadm_id, stay_id,
                   TRY_CAST(intime  AS TIMESTAMP) AS intime,
                   TRY_CAST(outtime AS TIMESTAMP) AS outtime,
                   TRY_CAST(los     AS DOUBLE)    AS los,
                   ROW_NUMBER() OVER (PARTITION BY subject_id
                                      ORDER BY TRY_CAST(intime AS TIMESTAMP)) AS rn
            FROM {rc(FP['icustays'])}
        ),
        pa AS (
            SELECT subject_id, TRY_CAST(anchor_age AS INT) AS anchor_age, gender
            FROM {rc(FP['patients'])}
        ),
        ad AS (
            SELECT hadm_id,
                   TRY_CAST(hospital_expire_flag AS INT) AS hospital_expire_flag,
                   TRY_CAST(deathtime AS TIMESTAMP)      AS deathtime,
                   admission_type
            FROM {rc(FP['admissions'])}
        )
        SELECT ic.subject_id, ic.hadm_id, ic.stay_id, ic.intime, ic.outtime, ic.los,
               pa.anchor_age, pa.gender,
               ad.hospital_expire_flag, ad.deathtime, ad.admission_type
        FROM ic
        JOIN pa ON ic.subject_id = pa.subject_id
        JOIN ad ON ic.hadm_id    = ad.hadm_id
        WHERE ic.rn = 1 AND ic.los >= {MIN_LOS_DAYS} AND pa.anchor_age >= {MIN_AGE}
    """).df()

    cohort["stay_id"] = cohort["stay_id"].astype(str)
    cohort["hadm_id"] = cohort["hadm_id"].astype(str)
    cohort = cohort.reset_index(drop=True)

    dt = pd.to_datetime(cohort["deathtime"], errors="coerce")
    ot = pd.to_datetime(cohort["outtime"],   errors="coerce")
    cohort["mortality_inhosp"] = cohort["hospital_expire_flag"].fillna(0).astype(int)
    cohort["icu_mortality"]    = ((dt.notna()) & (dt <= ot)).astype(int)
    cohort["long_stay"]        = (cohort["los"] >= LONG_STAY_DAYS).astype(int)

    TASKS = ["mortality_inhosp", "icu_mortality", "long_stay"]
    print(f"cohort N = {len(cohort):,}  (first ICU stay, age>={MIN_AGE}, LOS>={MIN_LOS_DAYS}d)")
    for t_ in TASKS:
        print(f"  {t_:18s} prevalence = {cohort[t_].mean():.4f}  (n_pos={int(cohort[t_].sum()):,})")
    stay2idx = {s: i for i, s in enumerate(cohort["stay_id"])}

    _age = cohort["anchor_age"].astype(float).values
    AGE_MEAN, AGE_STD = float(_age.mean()), float(_age.std() + 1e-6)
    _age_z = (_age - AGE_MEAN) / AGE_STD
    _is_female = (cohort["gender"].astype(str).str.upper() == "F").astype(float).values
    _at = cohort["admission_type"].astype(str).fillna("UNKNOWN")
    ADM_CATS = sorted(_at.unique().tolist())
    _onehot = np.stack([(_at == k).astype(float).values for k in ADM_CATS], axis=1)
    C_COV = np.concatenate([_age_z[:, None], _is_female[:, None], _onehot], axis=1).astype(np.float32)
    COV_NAMES = ["age_z", "is_female"] + [f"adm::{k}" for k in ADM_CATS]
    print(f"  covariates c: shape {C_COV.shape} (d_c={C_COV.shape[1]}) | cols: {COV_NAMES}")

## 6 · Stream the events inside the window

In [ ]:
if not COHORT_READY:
    con.register("cohort", cohort[["stay_id", "hadm_id", "intime"]])
    win = f"INTERVAL '{WINDOW_HOURS} hours'"
    chart_ids = ",".join(map(str, CHART_IDS))
    lab_ids   = ",".join(map(str, LAB_IDS))

    print("scanning chartevents ...")
    t0 = time.time()
    chart = con.execute(f"""
        SELECT c.stay_id AS stay_id,
               TRY_CAST(ce.itemid AS INT) AS itemid,
               CAST(floor(date_diff('minute', c.intime,
                     TRY_CAST(ce.charttime AS TIMESTAMP)) / 60.0) AS INT) AS hr,
               TRY_CAST(ce.valuenum AS DOUBLE) AS valuenum
        FROM read_csv('{FP['chartevents']}', header=true, sep=',', all_varchar=true,
                      compression='gzip', ignore_errors=true) ce
        JOIN cohort c ON ce.stay_id = c.stay_id
        WHERE TRY_CAST(ce.itemid AS INT) IN ({chart_ids})
          AND TRY_CAST(ce.charttime AS TIMESTAMP) >= c.intime
          AND TRY_CAST(ce.charttime AS TIMESTAMP) <  c.intime + {win}
    """).df()
    print(f"  chartevents rows kept: {len(chart):,}  ({time.time()-t0:.0f}s)")

    print("scanning labevents ...")
    t0 = time.time()
    lab = con.execute(f"""
        SELECT c.stay_id AS stay_id,
               TRY_CAST(le.itemid AS INT) AS itemid,
               CAST(floor(date_diff('minute', c.intime,
                     TRY_CAST(le.charttime AS TIMESTAMP)) / 60.0) AS INT) AS hr,
               TRY_CAST(le.valuenum AS DOUBLE) AS valuenum
        FROM read_csv('{FP['labevents']}', header=true, sep=',', all_varchar=true,
                      compression='gzip', ignore_errors=true) le
        JOIN cohort c ON le.hadm_id = c.hadm_id
        WHERE TRY_CAST(le.itemid AS INT) IN ({lab_ids})
          AND TRY_CAST(le.charttime AS TIMESTAMP) >= c.intime
          AND TRY_CAST(le.charttime AS TIMESTAMP) <  c.intime + {win}
    """).df()
    print(f"  labevents rows kept:   {len(lab):,}  ({time.time()-t0:.0f}s)")

    events = pd.concat([chart, lab], ignore_index=True)
    events = events[(events["hr"] >= 0) & (events["hr"] < T)]

    cov = events.groupby("itemid").size()
    print("\ncoverage (rows) per variable:")
    for name, cls, src, ids in PANEL:
        n = int(sum(cov.get(i, 0) for i in ids))
        flag = "  <-- LOW / check itemid" if n < 50 else ""
        print(f"  {name:16s} [{cls:9s}] itemids={ids} rows={n:,}{flag}")
    con.close()

## 7 · Assemble the (N, T, V) mask and value tensors, then cache

In [ ]:
if not COHORT_READY:
    N = len(cohort)
    events["i"] = events["stay_id"].map(stay2idx)
    events["v"] = events["itemid"].map(item2var)
    events = events.dropna(subset=["i", "v"]).copy()
    events["i"] = events["i"].astype(int); events["v"] = events["v"].astype(int)

    agg = (events.groupby(["i", "hr", "v"])["valuenum"]
                 .agg(["size", "mean"]).reset_index())
    m = np.zeros((N, T, V), np.int8)
    y = np.zeros((N, T, V), np.float32)
    ii, hh, vv = agg["i"].values, agg["hr"].values.astype(int), agg["v"].values
    m[ii, hh, vv] = 1
    y[ii, hh, vv] = np.nan_to_num(agg["mean"].values).astype(np.float32)

    labels_tbl = cohort[TASKS].reset_index(drop=True)
    save = dict(m=m, y=y, var_names=np.array(VAR_NAMES, dtype=object), var_class=VAR_CLASS,
                c=C_COV, cov_names=np.array(COV_NAMES, dtype=object),
                stay_id=cohort["stay_id"].values.astype(str),
                age_mean=np.float32(AGE_MEAN), age_std=np.float32(AGE_STD),
                adm_cats=np.array(ADM_CATS, dtype=object))
    for c in TASKS:
        save[f"lab_{c}"] = labels_tbl[c].values
    np.savez_compressed(CACHE_NPZ, **save)
    COHORT_READY = True
    print(f"tensors built: m,y shape = {m.shape}")
    print(f"  overall mask density = {m.mean():.4f}")
    print(f"cached -> {CACHE_NPZ}")

## 8 · The mask-channel audit

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit

def mask_features(m):
    N, Tt, Vv = m.shape
    count = m.sum(1).astype(np.float32)
    rate  = count / Tt
    obs   = m.astype(bool)
    first = np.where(obs.any(1), np.argmax(obs, 1).astype(np.float32), np.float32(Tt)) / Tt
    rev   = obs[:, ::-1, :]
    last_idx = np.where(obs.any(1),
                        (Tt - 1 - np.argmax(rev, 1)).astype(np.float32), np.float32(-1.0))
    last  = (last_idx + 1.0) / Tt
    return np.concatenate([count, rate, first, last], axis=1)

def mask_features_subset(m, var_idx):
    N, Tt, Vv = m.shape
    mf = mask_features(m)
    var_idx = np.asarray(var_idx, int)
    cols = np.concatenate([var_idx, Vv + var_idx, 2*Vv + var_idx, 3*Vv + var_idx])
    return mf[:, cols]

def value_features(y, m):
    N, Tt, Vv = y.shape
    obs = m.astype(bool)
    cnt = np.maximum(m.sum(1), 1)
    mean = (y.sum(1) / cnt).astype(np.float32)
    var  = np.maximum((y*y).sum(1) / cnt - mean**2, 0.0)
    std  = np.sqrt(var).astype(np.float32)
    any_obs = obs.any(1)
    vmin = np.where(any_obs, np.where(obs, y,  np.inf).min(1), 0.0).astype(np.float32)
    vmax = np.where(any_obs, np.where(obs, y, -np.inf).max(1), 0.0).astype(np.float32)
    rev  = obs[:, ::-1, :]
    last_t = np.where(any_obs, Tt - 1 - np.argmax(rev, 1), 0)
    last = np.take_along_axis(y, last_t[:, None, :], axis=1)[:, 0, :]
    last = np.where(any_obs, last, 0.0).astype(np.float32)
    feats = np.concatenate([mean, std, vmin, vmax, last], axis=1)
    return np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)

def auroc_cv(X, labels, seed=0):
    labels = np.asarray(labels)
    if len(np.unique(labels)) < 2:
        return np.array([0.5])
    aucs, sss = [], StratifiedShuffleSplit(N_SPLITS, test_size=TEST_FRAC, random_state=seed)
    for tr, te in sss.split(X, labels):
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=500, C=C_REG)
        clf.fit(sc.transform(X[tr]), labels[tr])
        aucs.append(roc_auc_score(labels[te], clf.decision_function(sc.transform(X[te]))))
    return np.array(aucs)

def ci(a):
    return float(np.mean(a)), float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))

## 9 · Featurise once

In [ ]:
assert COHORT_READY, "Run the extraction cells (4–7) or enable USE_CACHE with an existing cache."
t0 = time.time()
Xm = mask_features(m)
Xv = value_features(y, m)
Xj = np.concatenate([Xm, Xv], axis=1)
trig = np.where(VAR_CLASS == "triggered")[0]
prot = np.where(VAR_CLASS == "protocol")[0]
Xm_trig = mask_features_subset(m, trig)
Xm_prot = mask_features_subset(m, prot)
print(f"features: mask={Xm.shape} value={Xv.shape} joint={Xj.shape}  ({time.time()-t0:.0f}s)")

## 10 · Run the audit and print the GO/NO-GO verdict

In [ ]:
rows = []
TASKS = list(labels_tbl.columns)
for task in TASKS:
    lab = labels_tbl[task].values
    r = {"task": task, "prevalence": float(np.mean(lab))}
    r["mask"]  = ci(auroc_cv(Xm, lab))
    r["value"] = ci(auroc_cv(Xv, lab))
    r["joint"] = ci(auroc_cv(Xj, lab))
    rng = np.random.default_rng(0)
    r["placebo"] = ci(auroc_cv(Xm, rng.permutation(lab)))
    r["mask_triggered"] = ci(auroc_cv(Xm_trig, lab))
    r["mask_protocol"]  = ci(auroc_cv(Xm_prot, lab))
    r["GO"] = r["mask"][1] >= MARGIN
    rows.append(r)

def fmt(t): return f"{t[0]:.3f} [{t[1]:.3f},{t[2]:.3f}]"
print(f"{'task':18s} {'prev':>6s} | {'mask-only':>22s} {'value-only':>22s} {'joint':>22s} | verdict")
print("-"*110)
for r in rows:
    print(f"{r['task']:18s} {r['prevalence']:6.3f} | "
          f"{fmt(r['mask']):>22s} {fmt(r['value']):>22s} {fmt(r['joint']):>22s} | "
          f"{'GO ' if r['GO'] else 'no-go'}")

print("\nControls (mask channel):")
print(f"{'task':18s} | {'placebo(perm)':>16s} | {'triggered-mask':>16s} {'protocol-mask':>16s}")
print("-"*80)
for r in rows:
    tri, pro = r['mask_triggered'][0], r['mask_protocol'][0]
    order = "triggered > protocol OK" if tri > pro else "triggered <= protocol (!)"
    print(f"{r['task']:18s} | {r['placebo'][0]:16.3f} | {tri:16.3f} {pro:16.3f}   {order}")

headline = [r['task'] for r in rows if r['GO']]
print("\n" + "="*70)
print("VIABLE HEADLINE TASKS:", headline if headline else "NONE (revisit panel/horizon)")
print("Interpretation: placebo should sit ~0.50 (no leakage); a GO task with "
      "mask << joint\nand triggered-mask carrying signal is exactly the §4b setting where "
      "a mask-agnostic\ngenerator should fail and RecordDiff should not.")
pd.DataFrame([{**{'task': r['task'], 'prevalence': round(r['prevalence'],3)},
               **{k: round(r[k][0],3) for k in ['mask','value','joint','placebo',
                                                'mask_triggered','mask_protocol']},
               'GO': r['GO']} for r in rows])